# Profundidad entre sectores con todas las fotometrías disponibles

**Kernel: `starry`.** No depende de TESScut: usa las curvas ya publicadas en MAST
(`Light_Curve_Files_MAST/`) más las TGLC (HLSP de MAST en S5/S6 y, si ya se corrió `1c`, las locales).

Pregunta: el cambio de la profundidad entre sectores, ¿es una variación real del flujo de la estrella o
un efecto sistemático? Si es real, **todas las fotometrías independientes** deberían ver el mismo patrón
entre sectores. Si es sistemático de la fotometría tess_extractor, las demás no lo ven.

| fotometría | qué es | corrige dilución |
|---|---|---|
| `tess_extractor` | `Flux_CBV_SS` de `Light_Curve_Files/` (curvas de tess_extractor) | ? |
| `TGLC` | apertura 3×3 sobre la imagen descontaminada con Gaia (`cal_aper_flux`); en S≥94 se usa la curva de QLP, que desde S94 **es** fotometría TGLC 3×3 (Petitpas et al. 2026) | sí (modelo de la escena con Gaia) |
| `QLP` | apertura FIPHOT del MIT Quick Look Pipeline (`SAP_FLUX`), **solo S<94** | no |
| `CDIPS` | imagen diferencia para cúmulos jóvenes (`IRM2`, apertura 2) | no |
| `PDCSAP` | SPOC / TESS-SPOC (`PDCSAP_FLUX`) | sí, con `CROWDSAP` del TIC |

Como cada fotometría trata la contaminación distinto, sus profundidades **absolutas** no tienen que
coincidir. Lo que se compara es el **patrón entre sectores**: $D(s)/\langle D\rangle$ dentro de cada
fotometría.

Todo se mide con las funciones del 1b (`medir`) y del 1d (`eclipses_individuales`: profundidad de cada
eclipse en su propio mínimo, sin el achatamiento por efeméride), con las efemérides de `priors_v2.csv`.

In [ ]:
import json
import ast
import glob
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from astropy.io import fits

RUTA_ORION = Path("/home/luciam/Documents/starry/Resultados_Orion")
DIR_MAST = RUTA_ORION / "Light_Curve_Files_MAST"
DIR_TGLC_HLSP = RUTA_ORION / "Light_Curve_Files_TGLC"
DIR_TGLC_LOCAL = DIR_TGLC_HLSP / "local"
DIR_FIG = RUTA_ORION / "resultados_pipeline" / "profundidades_v2" / "pipelines"
DIR_FIG.mkdir(parents=True, exist_ok=True)


def cargar_de_notebook(nb_path, celdas=None, nombres=(), funciones=None):
    # ejecuta de un notebook solo imports, funciones (todas o las de `funciones`) y las asignaciones en `nombres`
    nb = json.load(open(nb_path))
    idx = celdas if celdas is not None else [i for i, c in enumerate(nb["cells"]) if c["cell_type"] == "code"]
    for i in idx:
        try:
            arbol = ast.parse("".join(nb["cells"][i]["source"]))
        except SyntaxError:
            continue
        nodos = []
        for n in arbol.body:
            if isinstance(n, (ast.Import, ast.ImportFrom)):
                nodos.append(n)
            elif isinstance(n, ast.FunctionDef) and (funciones is None or n.name in funciones):
                nodos.append(n)
            elif isinstance(n, ast.Assign):
                ids = [t.id for t in n.targets if isinstance(t, ast.Name)]
                ids += [e.id for t in n.targets if isinstance(t, ast.Tuple) for e in t.elts]
                if any(x in nombres for x in ids):
                    nodos.append(n)
        exec(compile(ast.Module(body=nodos, type_ignores=[]), f"{Path(nb_path).name}[{i}]", "exec"), globals())


NB_1B = RUTA_ORION / "1b_profundidades_eclipse_v2.ipynb"
cargar_de_notebook(NB_1B, [4], ("RUTA_CURVAS", "WINDOW_WOTAN", "ALFAS", "N_MIN", "W_MAX", "TOL_P"))
cargar_de_notebook(NB_1B, [5], ("ALFA_0",))
cargar_de_notebook(NB_1B, [17], ("P_UMBRAL", "F_UMBRAL", "csv_idx", "INDICE"), funciones=("consistencia",))
cargar_de_notebook(NB_1B, [20], ("COLOR_SECTOR",), funciones=("mediana_bines",))
cargar_de_notebook(RUTA_ORION / "1d_tglc_profundidades.ipynb",
                   funciones=("eclipses_individuales", "estadisticos"))

v2 = pd.read_csv(RUTA_ORION / "priors_v2.csv")
cons_1b = pd.read_csv(RUTA_ORION / "resultados_pipeline" / "profundidades_v2" / "consistencia_sectores.csv")
SOLO_REVISAR = True   # True: los TIC con alguna marca distinta de "consistente" en el 1b; False: todos
if SOLO_REVISAR:
    TICS = cons_1b.loc[(cons_1b["marca_primario"] != "consistente")
                       | (cons_1b["marca_secundario"] == "difiere_revisar"), "TIC"].astype(int).tolist()
else:
    TICS = sorted(v2["TIC"].astype(int).unique())
print(len(TICS), "sistemas")

## 1) Cargadores

Cada cargador devuelve `(t, f)` con las banderas de calidad de su pipeline aplicadas, o `None` si no
hay curva. En TGLC se usa la curva local (`1c`) si existe y, si no, el HLSP de MAST; en TIC 264741359
(dos fuentes Gaia a 1.3") se toma la más brillante, la misma que elige `1c`.

In [ ]:
def _limpio(t, f):
    t, f = np.asarray(t, float), np.asarray(f, float)
    ok = np.isfinite(t) & np.isfinite(f) & (f > 0)
    t, f = t[ok], f[ok]
    o = np.argsort(t)
    t, f = t[o], f[o]
    u = np.r_[True, np.diff(t) > 0]
    return (t[u], f[u]) if u.sum() >= 100 else None


def cargar_tess_extractor(tic, s):
    p = RUTA_CURVAS / f"tic {tic}_S_{s}_LC.txt"
    return cargar_curva(tic, s) if p.exists() else None


def cargar_tglc(tic, s, columna="cal_aper_flux"):
    # desde S94 la curva de QLP es fotometría TGLC 3x3 (Petitpas et al. 2026): se usa como TGLC
    if s >= 94:
        return _cargar_qlp(tic, s)
    p = DIR_TGLC_LOCAL / f"tic {tic}_S_{s}_TGLC.txt"
    if p.exists():
        d = pd.read_csv(p, sep=" ", comment="#")
        d = d.loc[(d["TGLC_flags"] == 0) & (d["TESS_flags"] == 0)]
        return _limpio(d["time"], d[columna])
    cand = []
    for f in glob.glob(str(DIR_TGLC_HLSP / f"*-s{s:04d}-*_llc.fits")):
        h = fits.open(f)
        if str(h[0].header["TICID"]) == str(tic):
            cand.append((h[0].header["TESSMAG"], h))
    if not cand:
        return None
    h = sorted(cand, key=lambda x: x[0])[0][1]
    d = h[1].data
    ok = (d["TGLC_flags"] == 0) & (d["TESS_flags"] == 0)
    return _limpio(d["time"][ok], d[columna][ok])


def cargar_qlp(tic, s):
    # QLP clásico (FIPHOT, sin descontaminar) solo hasta S93
    return _cargar_qlp(tic, s) if s < 94 else None


def _cargar_qlp(tic, s):
    f = DIR_MAST / f"hlsp_qlp_tess_ffi_s{s:04d}-{tic:016d}_tess_v01_llc.fits"
    if not f.exists():
        return None
    d = fits.open(f)[1].data
    ok = d["QUALITY"] == 0
    return _limpio(d["TIME"][ok], d["SAP_FLUX"][ok])


def _gaia_de_tic(tic):
    # el nombre de los archivos CDIPS usa el Gaia DR2; se toma de la cabecera de TGLC/QLP o del TIC
    for f in glob.glob(str(DIR_MAST / "hlsp_cdips_*")):
        h = fits.getheader(f, 0)
        if str(h.get("TICID", "")).lstrip("0") == str(tic):
            yield f


def cargar_cdips(tic, s):
    for f in _gaia_de_tic(tic):
        if f"-s{s:04d}-" in f:
            d = fits.open(f)[1].data
            ok = np.asarray(d["IRQ2"]) == "G"
            return _limpio(d["TMID_BJD"][ok] - 2457000.0, 10 ** (-0.4 * d["IRM2"][ok]))
    return None


def cargar_pdcsap(tic, s):
    # preferencia: SPOC 2 min; si no hay, TESS-SPOC (FFI)
    pats = [f"tess*-s{s:04d}-{tic:016d}-*-s_lc.fits", f"hlsp_tess-spoc_tess_phot_{tic:016d}-s{s:04d}_tess_v1_lc.fits"]
    for pat in pats:
        fs = glob.glob(str(DIR_MAST / pat))
        if fs:
            d = fits.open(fs[0])[1].data
            ok = d["QUALITY"] == 0
            return _limpio(d["TIME"][ok], d["PDCSAP_FLUX"][ok])
    return None


FOTOMETRIAS = {"tess_extractor": cargar_tess_extractor, "TGLC": cargar_tglc, "QLP": cargar_qlp, "CDIPS": cargar_cdips,
               "PDCSAP": cargar_pdcsap}
COLOR_FOT = {"tess_extractor": "#2a78d6", "TGLC": "black", "QLP": "#eb6834", "CDIPS": "#1baf7a", "PDCSAP": "#8e44ad"}
MARCA_FOT = {"tess_extractor": "o", "TGLC": "x", "QLP": "s", "CDIPS": "^", "PDCSAP": "D"}

disp = []
for r in v2.loc[v2["TIC"].isin(TICS)].itertuples():
    for fot, carg in FOTOMETRIAS.items():
        if carg(int(r.TIC), int(r.Sector)) is not None:
            disp.append((int(r.TIC), int(r.Sector), fot))
disp = pd.DataFrame(disp, columns=["TIC", "Sector", "fotometria"])
disp.pivot_table(index="TIC", columns="fotometria", values="Sector", aggfunc=lambda x: ",".join(map(str, sorted(x))))

## 2) Medición por sector y por eclipse

Para cada (TIC, sector, fotometría): `medir` del 1b con la efeméride de `priors_v2.csv`, y luego la
profundidad de cada eclipse (`eclipses_individuales` del 1d). La profundidad de referencia de un sector
es la **mediana de sus eclipses individuales** (`D_ind`), que no se achata si la efeméride está un poco
corrida.

In [ ]:
filas, filas_ind, curvas_all = [], [], {}
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    for (tic, s, fot) in disp.itertuples(index=False):
        r = v2.loc[(v2["TIC"] == tic) & (v2["Sector"] == s)].iloc[0]
        t, f = FOTOMETRIAS[fot](tic, s)
        m, _, _ = medir(t, f, r["P_orb"], r["t0"], r["Duration"])
        fila = dict(TIC=tic, Sector=s, fotometria=fot, P=r["P_orb"], t0=r["t0"], Duration=r["Duration"],
                    texp_sobre_duracion=np.median(np.diff(t)) / r["Duration"], **m)
        filas.append(fila)
        curvas_all[(fot, tic, s)] = (t, f)
        if not m["eclipse_ancho"]:
            for e in eclipses_individuales(t, f, pd.Series(fila)):
                filas_ind.append(dict(TIC=tic, Sector=s, fotometria=fot, **e))
med_all = pd.DataFrame(filas)
ind_all = pd.DataFrame(filas_ind)
ind_all.to_csv(DIR_FIG / "eclipses_individuales_todas.csv", index=False)

agg = (ind_all.groupby(["TIC", "Sector", "fotometria", "eclipse"])["D"]
       .agg(D_ind="median", n="count", s="std").reset_index())
agg["D_ind_err"] = 1.2533 * agg["s"] / np.sqrt(agg["n"])   # error de la mediana
tabla = med_all[["TIC", "Sector", "fotometria", "D_primario", "D_secundario", "secundario_detectado",
                 "texp_sobre_duracion"]].merge(
    agg.loc[agg["eclipse"] == "primario", ["TIC", "Sector", "fotometria", "D_ind", "D_ind_err", "n"]]
    .rename(columns={"D_ind": "D1_ind", "D_ind_err": "D1_ind_err", "n": "n1"}), how="left").merge(
    agg.loc[agg["eclipse"] == "secundario", ["TIC", "Sector", "fotometria", "D_ind", "D_ind_err"]]
    .rename(columns={"D_ind": "D2_ind", "D_ind_err": "D2_ind_err"}), how="left")
tabla["D2/D1"] = tabla["D2_ind"] / tabla["D1_ind"]
tabla.to_csv(DIR_FIG / "profundidades_por_fotometria.csv", index=False)
tabla.pivot_table(index=["TIC", "Sector"], columns="fotometria", values="D1_ind").round(4)

## 3) Patrón entre sectores

Para cada fotometría se divide `D1_ind` por su promedio en los sectores que tiene. Si el cambio entre
sectores es real, las curvas de todas las fotometrías se superponen; si es sistemático de una, esa se
separa de las demás. El panel derecho muestra D2/D1, que no depende de la dilución.

In [ ]:
def norm_patron(g, col):
    g = g.dropna(subset=[col])
    if len(g) < 2:
        return pd.Series(dtype=float), pd.Series(dtype=float)
    w = 1 / g[f"{col}_err"].clip(lower=1e-6) ** 2
    m = np.sum(w * g[col]) / w.sum()
    return (g.set_index("Sector")[col] / m), (g.set_index("Sector")[f"{col}_err"] / m)


def figura_patron(tic, axs):
    g = tabla.loc[tabla["TIC"] == tic]
    secs = sorted(g["Sector"].unique())
    xpos = {s: k for k, s in enumerate(secs)}
    fots = [f for f in FOTOMETRIAS if f in g["fotometria"].values]
    for j, fot in enumerate(fots):
        h = g.loc[g["fotometria"] == fot].sort_values("Sector")
        y, e = norm_patron(h, "D1_ind")
        dx = (j - (len(fots) - 1) / 2) * 0.07
        if len(y):
            xs = [xpos[s] + dx for s in y.index]
            axs[0].errorbar(xs, y.values, yerr=e.values, fmt=MARCA_FOT[fot] + "-", color=COLOR_FOT[fot], ms=6,
                            lw=1.3, capsize=2, label=fot, mfc="none" if fot == "TGLC" else COLOR_FOT[fot])
        h2 = h.dropna(subset=["D2/D1"])
        if len(h2):
            axs[1].plot([xpos[s] + dx for s in h2["Sector"]], h2["D2/D1"], MARCA_FOT[fot] + "-",
                        color=COLOR_FOT[fot], ms=6, lw=1.3, mfc="none" if fot == "TGLC" else COLOR_FOT[fot])
    for ax in axs:
        ax.set_xticks(range(len(secs)))
        ax.set_xticklabels([f"S{s}" for s in secs])
        ax.set_xlim(-0.5, len(secs) - 0.5)
        ax.grid(alpha=0.25)
        ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)
    axs[0].axhline(1, color="0.6", lw=0.8)
    axs[0].set_ylabel("D1 / promedio de la fotometría")
    axs[1].set_ylabel("D2 / D1")
    idx = INDICE.get(tic, np.nan)
    axs[0].set_title((f"sistema {int(idx)} · " if np.isfinite(idx) else "") + f"TIC {tic}", loc="left", fontsize=10)


orden = sorted(TICS, key=lambda t: INDICE.get(t, 99))
nfil = int(np.ceil(len(orden) / 2))
fig, axs = plt.subplots(nfil, 4, figsize=(20, 3.6 * nfil), squeeze=False,
                        gridspec_kw={"hspace": 0.55, "wspace": 0.32})
for k, tic in enumerate(orden):
    figura_patron(tic, axs[k // 2][2 * (k % 2): 2 * (k % 2) + 2])
for k in range(len(orden), 2 * nfil):
    for ax in axs[k // 2][2 * (k % 2): 2 * (k % 2) + 2]:
        ax.axis("off")
from matplotlib.lines import Line2D
fig.legend(handles=[Line2D([], [], color=COLOR_FOT[f], marker=MARCA_FOT[f], label=f,
                           mfc="none" if f == "TGLC" else COLOR_FOT[f]) for f in FOTOMETRIAS],
           loc="upper center", ncol=5, frameon=False, fontsize=11, bbox_to_anchor=(0.5, 1 - 0.25 / fig.get_figheight()))
fig.subplots_adjust(top=1 - 1.1 / fig.get_figheight())
fig.suptitle("Patrón de la profundidad entre sectores por fotometría (mediana de eclipses individuales)",
             y=1 - 0.05 / fig.get_figheight(), fontsize=13)
fig.savefig(DIR_FIG / "patron_sectores_fotometrias.png", dpi=110, bbox_inches="tight")
plt.show()

## 4) Veredicto por sistema

Para cada fotometría con al menos 2 sectores se calcula, con los eclipses individuales:
- `p_entre`: ANOVA, ¿las medias de los sectores difieren más que la dispersión entre eclipses?
- `rho_brillo` / `p_brillo`: correlación entre D y el brillo fuera de eclipse dentro de cada sector
  (firma de manchas).

Y para cada fotometría independiente, si su patrón $D(s)/\langle D\rangle$ **coincide con el de la
tess_extractor**: diferencia en cada sector común menor que 3σ (σ combinado de las dos) **o** menor que 2 %.

Reglas, en orden:
1. tess_extractor no tiene salto entre sectores (`p_entre ≥ 0.0027`) → **consistente** (lo que marcaba el 1b
   era el achatamiento por efeméride o el estimador por sector).
2. Ninguna otra fotometría con 2+ sectores en común → **sin comparación independiente**.
3. Todas las demás reproducen el patrón y muestran salto → **real** (variación de la estrella o de su
   entorno); si además D correlaciona con el brillo → **real: manchas**.
4. Ninguna lo reproduce → **sistemático de la fotometría tess_extractor**.
5. Mixto → **revisar a ojo** (las figuras de la §5).

In [ ]:
est_all = pd.DataFrame([dict(TIC=k[0], fotometria=k[1], eclipse=k[2], **estadisticos(g))
                        for k, g in ind_all.groupby(["TIC", "fotometria", "eclipse"])])
est_all.to_csv(DIR_FIG / "estadisticos_por_fotometria.csv", index=False)


def coincide(tic, fot):
    g = tabla.loc[tabla["TIC"] == tic]
    yp, ep = norm_patron(g.loc[g["fotometria"] == "tess_extractor"].sort_values("Sector"), "D1_ind")
    yo, eo = norm_patron(g.loc[g["fotometria"] == fot].sort_values("Sector"), "D1_ind")
    comunes = sorted(set(yp.index) & set(yo.index))
    if len(comunes) < 2:
        return np.nan, comunes
    # renormaliza ambas en los sectores comunes antes de comparar
    yp, yo = yp[comunes] / yp[comunes].mean(), yo[comunes] / yo[comunes].mean()
    sig = np.hypot(ep[comunes], eo[comunes])
    dif = (yp - yo).abs()
    return bool(np.all((dif < 3 * sig) | (dif < 0.02))), comunes


def veredicto(tic):
    e = est_all.loc[(est_all["TIC"] == tic) & (est_all["eclipse"] == "primario")].set_index("fotometria")
    fila = dict(TIC=tic, indice=INDICE.get(tic, np.nan))
    if "tess_extractor" not in e.index:
        return dict(fila, veredicto="sin datos propios")
    pp = e.loc["tess_extractor"].get("p_entre", np.nan)
    fila["p_entre_tess_extractor"] = pp
    fila["p_brillo_tess_extractor"] = e.loc["tess_extractor"].get("p_brillo", np.nan)
    otras = {}
    for fot in e.index:
        if fot == "tess_extractor":
            continue
        c, comunes = coincide(tic, fot)
        if not isinstance(c, float):
            otras[fot] = dict(coincide=c, p_entre=e.loc[fot].get("p_entre", np.nan),
                              p_brillo=e.loc[fot].get("p_brillo", np.nan), sectores=comunes)
    fila["otras"] = "; ".join(f"{k}: {'sí' if v['coincide'] else 'no'} coincide (p_entre={v['p_entre']:.1e}, "
                              f"S{','.join(map(str, v['sectores']))})" for k, v in otras.items())
    manchas = [x < P_UMBRAL for x in [fila["p_brillo_tess_extractor"]] + [v["p_brillo"] for v in otras.values()]
               if np.isfinite(x)]
    if not np.isfinite(pp):
        v = "un solo sector medible en tess_extractor"
    elif pp >= P_UMBRAL:
        v = "consistente con eclipses individuales (el salto del 1b no se confirma)"
    elif not otras:
        v = "salto en tess_extractor; sin otra fotometría con 2+ sectores en común"
    elif all(x["coincide"] for x in otras.values()):
        v = "REAL: todas las fotometrías ven el mismo cambio"
        if manchas and sum(manchas) >= max(1, len(manchas) // 2):
            v += " (D correlaciona con el brillo: manchas)"
    elif not any(x["coincide"] for x in otras.values()):
        v = "SISTEMÁTICO de la fotometría tess_extractor: ninguna otra ve el cambio"
    else:
        v = "mixto: unas fotometrías ven el cambio y otras no (revisar)"
    fila["veredicto"] = v
    return fila


ver = pd.DataFrame([veredicto(t) for t in orden])
ver.to_csv(DIR_FIG / "veredicto_por_sistema.csv", index=False)
pd.set_option("display.max_colwidth", 200)
ver

## 5) Curvas plegadas: una fila por sistema, una columna por fotometría

Eclipse primario, cada sector con su color (como en `plegadas_sectores_todos.png`). Así se ve si el
sector que se separa en tess_extractor también se separa en las demás. Las líneas discontinuas son
$1-D_{\rm ind}$ de cada sector.

In [ ]:
def panel_plegado(ax, fot, tic):
    g = med_all.loc[(med_all["TIC"] == tic) & (med_all["fotometria"] == fot) & ~med_all["eclipse_ancho"]]
    if g.empty:
        ax.axis("off")
        return
    dur_h = g["Duration"].median() * 24
    rango = 0.85 * dur_h
    lo = 1.0
    for _, r in g.sort_values("Sector").iterrows():
        t, f = curvas_all[(fot, tic, r["Sector"])]
        phi2 = r["fase_secundario"] if r["secundario_detectado"] else 0.5
        F, _ = normalizar_sin_eclipses(t, f, r["t0"], r["P"], r["W"], phi2)
        ph = fase(t, r["t0"], r["P"])
        c1 = r["centro_primario"] if np.isfinite(r["centro_primario"]) else 0.0
        h = (((ph - c1) + 0.5) % 1 - 0.5) * r["P"] * 24
        col = COLOR_SECTOR.get(int(r["Sector"]), "0.4")
        sel = np.abs(h) < rango
        ax.plot(h[sel], F[sel], ".", ms=1.2, color=col, alpha=0.15)
        ancho = max(dur_h / 30, 1.2 * np.median(np.diff(t)) * 24)
        xb, yb = mediana_bines(h, F, ancho, rango)
        ax.plot(xb, yb, "-", color=col, lw=2, label=f"S{int(r['Sector'])}")
        D = tabla.loc[(tabla["TIC"] == tic) & (tabla["Sector"] == r["Sector"]) & (tabla["fotometria"] == fot), "D1_ind"]
        if len(D) and np.isfinite(D.iloc[0]):
            ax.axhline(1 - D.iloc[0], color=col, ls="--", lw=1)
            lo = min(lo, 1 - D.iloc[0])
    ax.axhline(1, color="0.6", lw=0.8)
    ax.set_xlim(-rango, rango)
    ax.set_ylim(1 - 1.25 * (1 - lo) - 0.003, 1 + 0.2 * (1 - lo) + 0.003)
    ax.legend(fontsize=7, loc="lower left", frameon=False)
    ax.grid(alpha=0.2)
    ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)


fots = list(FOTOMETRIAS)
fig, axs = plt.subplots(len(orden), len(fots), figsize=(4.2 * len(fots), 3.2 * len(orden)), squeeze=False,
                        gridspec_kw={"hspace": 0.5, "wspace": 0.25})
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    for i, tic in enumerate(orden):
        for j, fot in enumerate(fots):
            panel_plegado(axs[i][j], fot, tic)
        idx = INDICE.get(tic, np.nan)
        v = ver.loc[ver["TIC"] == tic, "veredicto"]
        axs[i][0].set_title((f"sistema {int(idx)} · " if np.isfinite(idx) else "") + f"TIC {tic}"
                            + (f"\n{v.iloc[0]}" if len(v) else ""), loc="left", fontsize=9)
for j, fot in enumerate(fots):
    axs[0][j].text(0.5, 1.45, fot, transform=axs[0][j].transAxes, ha="center", fontsize=13, color=COLOR_FOT[fot])
fig.savefig(DIR_FIG / "plegadas_por_fotometria.png", dpi=100, bbox_inches="tight")
plt.show()

## 6) ¿Es la cadencia? Todas las curvas llevadas a 30 min

El orden de los sectores (S5/S6 → S32/S33 → S87/S98) coincide con el de las cadencias (30 → 10 → 3.3
min). Una cadencia larga achata los eclipses cortos (más los en V que los de fondo plano), así que un
cambio de D o de D2/D1 que siga ese orden podría ser solo cadencia. Aquí se promedian todas las curvas a
30 min (como S5/S6) y se vuelve a medir: lo que **sobrevive** a cadencia común no es efecto de la
cadencia.

In [ ]:
def binear_cadencia(t, f, minutos):
    w = minutos / 1440
    k = np.floor((t - t[0]) / w)
    df = pd.DataFrame(dict(k=k, t=t, f=f)).groupby("k").agg(t=("t", "mean"), f=("f", "mean"), n=("f", "size"))
    df = df.loc[df["n"] >= max(1, 0.6 * w / np.median(np.diff(t)))]
    return df["t"].to_numpy(), df["f"].to_numpy()


filas30 = []
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    for (tic, s, fot) in disp.itertuples(index=False):
        r = v2.loc[(v2["TIC"] == tic) & (v2["Sector"] == s)].iloc[0]
        t, f = curvas_all[(fot, tic, s)]
        if np.median(np.diff(t)) * 1440 < 20:
            t, f = binear_cadencia(t, f, 30)
        m, _, _ = medir(t, f, r["P_orb"], r["t0"], r["Duration"])
        if m["eclipse_ancho"]:
            continue
        fila = dict(P=r["P_orb"], t0=r["t0"], Duration=r["Duration"], **m)
        e = pd.DataFrame(eclipses_individuales(t, f, pd.Series(fila)))
        if e.empty:
            continue
        d1 = e.loc[e["eclipse"] == "primario", "D"]
        d2 = e.loc[e["eclipse"] == "secundario", "D"]
        filas30.append(dict(TIC=tic, Sector=s, fotometria=fot, D1_30=d1.median(),
                            D1_30_err=1.2533 * d1.std() / np.sqrt(len(d1)) if len(d1) > 1 else np.nan,
                            D2_30=d2.median() if len(d2) else np.nan))
t30 = pd.DataFrame(filas30)
t30["D2/D1_30"] = t30["D2_30"] / t30["D1_30"]
t30.to_csv(DIR_FIG / "profundidades_cadencia_30min.csv", index=False)
comp30 = tabla[["TIC", "Sector", "fotometria", "D1_ind", "D2/D1"]].merge(t30, on=["TIC", "Sector", "fotometria"])
comp30.loc[comp30["fotometria"].isin(["tess_extractor", "QLP", "CDIPS", "PDCSAP"])].pivot_table(
    index=["TIC", "Sector"], columns="fotometria", values=["D1_30", "D2/D1_30"]).round(4)

In [ ]:
# mismo patrón entre sectores que en la §3, ahora a cadencia común de 30 min
fig, axs = plt.subplots(nfil, 4, figsize=(20, 3.6 * nfil), squeeze=False,
                        gridspec_kw={"hspace": 0.55, "wspace": 0.32})
tabla_orig = tabla
tabla = t30.rename(columns={"D1_30": "D1_ind", "D1_30_err": "D1_ind_err", "D2/D1_30": "D2/D1"})
for k, tic in enumerate(orden):
    figura_patron(tic, axs[k // 2][2 * (k % 2): 2 * (k % 2) + 2])
for k in range(len(orden), 2 * nfil):
    for ax in axs[k // 2][2 * (k % 2): 2 * (k % 2) + 2]:
        ax.axis("off")
tabla = tabla_orig
fig.legend(handles=[Line2D([], [], color=COLOR_FOT[f], marker=MARCA_FOT[f], label=f,
                           mfc="none" if f == "TGLC" else COLOR_FOT[f]) for f in FOTOMETRIAS],
           loc="upper center", ncol=5, frameon=False, fontsize=11, bbox_to_anchor=(0.5, 1 - 0.25 / fig.get_figheight()))
fig.subplots_adjust(top=1 - 1.1 / fig.get_figheight())
fig.suptitle("Patrón entre sectores con todas las curvas a 30 min (sin efecto de cadencia)",
             y=1 - 0.05 / fig.get_figheight(), fontsize=13)
fig.savefig(DIR_FIG / "patron_sectores_fotometrias_30min.png", dpi=110, bbox_inches="tight")
plt.show()